# Clase 2 · Laboratorio — Diseño dimensional para Saber 11

**Trabajo en parejas · 90 min.**

Al final deben tener un notebook completado con las 4 tareas resueltas y subirlo a Moodle antes de las 23:59.

**Checkpoint conjunto a los 55 min:** el profesor detiene la sala y revisamos juntos la Tarea 3 (hecho + FKs) antes de pasar al diseño del proyecto propio.

In [9]:
import pandas as pd

df = pd.read_csv("datos/saber11_muestra_500k.csv")
print(f"Cargados {len(df):,} registros — {df.shape[1]} columnas")

Cargados 500,000 registros — 22 columnas


## Tarea 1 (15 min) — dim_colegio con clave surrogate

Construye una tabla de dimensión `dim_colegio` que contenga:
- Una clave surrogate `colegio_id` (entero secuencial).
- Los atributos: COLE_NATURALEZA, COLE_JORNADA, COLE_CALENDARIO, COLE_BILINGUE.

Requisitos:
- Sin duplicados (una fila por combinación única de los 4 atributos).
- Verifica que la clave surrogate es única.

In [10]:
# Construir la dimensión de colegios con una fila por combinación única
atributos_colegio = [
    "COLE_NATURALEZA",
    "COLE_JORNADA",
    "COLE_CALENDARIO",
    "COLE_BILINGUE",
]
dim_colegio = df[atributos_colegio].drop_duplicates().reset_index(drop=True)
dim_colegio.insert(0, "colegio_id", dim_colegio.index + 1)

# Validar unicidad de la surrogate
assert dim_colegio["colegio_id"].is_unique, "La clave surrogate NO es única"
print(f"dim_colegio: {len(dim_colegio)} filas")
dim_colegio.head()

dim_colegio: 40 filas


,colegio_id,COLE_NATURALEZA,COLE_JORNADA,COLE_CALENDARIO,COLE_BILINGUE
0,1,OFICIAL,COMPLETA,A,N
1,2,NO OFICIAL,NOCHE,A,N
2,3,OFICIAL,TARDE,A,N
3,4,OFICIAL,MAÑANA,A,N
4,5,NO OFICIAL,TARDE,A,N


## Tarea 2 (15 min) — dim_geografia con jerarquía

Construye `dim_geografia` con:
- Clave surrogate `geo_id`.
- Atributos: COLE_DEPTO_UBICACION (padre), COLE_MCPIO_UBICACION (hijo).

Requisitos:
- Una fila por par (departamento, municipio).
- Documenta en Markdown por qué la jerarquía es útil para el análisis.

La jerarquía permite analizar los resultados en distintos niveles de detalle. Primero se pueden comparar departamentos y luego profundizar en los municipios de cada departamento. Esto facilita identificar diferencias geográficas, resumir indicadores y localizar zonas que requieren un análisis más específico.

In [11]:
# Construir la dimensión geográfica con una fila por departamento y municipio
atributos_geografia = [
    "COLE_DEPTO_UBICACION",
    "COLE_MCPIO_UBICACION",
]
dim_geografia = (
    df[atributos_geografia]
    .drop_duplicates()
    .reset_index(drop=True)
)
dim_geografia.insert(0, "geo_id", dim_geografia.index + 1)

# Validar la clave surrogate y la unicidad de la combinación geográfica
assert dim_geografia["geo_id"].is_unique, "La clave surrogate NO es única"
assert not dim_geografia[atributos_geografia].duplicated().any(), (
    "Hay pares departamento-municipio duplicados"
)
print(f"dim_geografia: {len(dim_geografia)} filas")
dim_geografia.head()

dim_geografia: 10 filas


,geo_id,COLE_DEPTO_UBICACION,COLE_MCPIO_UBICACION
0,1,ATLÁNTICO,BARRANQUILLA
1,2,BOLÍVAR,CARTAGENA
2,3,BOGOTÁ D.C.,BOGOTÁ D.C.
3,4,NARIÑO,PASTO
4,5,CÓRDOBA,MONTERÍA


## Tarea 3 (25 min) — hecho_resultados

Construye la tabla de hechos `hecho_resultados` con:
- FKs a: `dim_colegio` (colegio_id), `dim_geografia` (geo_id), `dim_tiempo` (tiempo_id — bosqueja también esta dimensión).
- Medidas: PUNT_LECTURA_CRITICA, PUNT_MATEMATICAS, PUNT_C_NATURALES, PUNT_SOCIALES_CIUDADANAS, PUNT_INGLES, PUNT_GLOBAL.

Requisitos:
- La granularidad del hecho es "una fila por estudiante-periodo".
- Después de hacer los joins con las dimensiones, la tabla de hechos NO debe perder filas (comparar `len(hecho_resultados)` con `len(df)` original).

In [12]:
# 1) dim_tiempo (boceto)
dim_tiempo = df[["PERIODO"]].drop_duplicates().reset_index(drop=True)
dim_tiempo["tiempo_id"] = dim_tiempo.index + 1

medidas = [
    "PUNT_LECTURA_CRITICA",
    "PUNT_MATEMATICAS",
    "PUNT_C_NATURALES",
    "PUNT_SOCIALES_CIUDADANAS",
    "PUNT_INGLES",
    "PUNT_GLOBAL",
]

hecho_resultados = (
    df
    .merge(dim_colegio, on=atributos_colegio, how="left")
    .merge(dim_geografia, on=atributos_geografia, how="left")
    .merge(dim_tiempo, on="PERIODO", how="left")
)

# Nos quedamos solo con las FKs + las medidas (esquema estrella)
hecho_resultados = hecho_resultados[["colegio_id", "geo_id", "tiempo_id"] + medidas]

# 3) Validar que no perdimos filas y que no hay FKs nulas
assert len(hecho_resultados) == len(df), f"Perdimos filas: {len(df) - len(hecho_resultados)}"
assert hecho_resultados[["colegio_id", "geo_id", "tiempo_id"]].notna().all().all(), "Hay FKs nulas"
print(f"hecho_resultados: {len(hecho_resultados)} filas (esperado {len(df)})")
hecho_resultados.head()

hecho_resultados: 500000 filas (esperado 500000)


,colegio_id,geo_id,tiempo_id,PUNT_LECTURA_CRITICA,PUNT_MATEMATICAS,PUNT_C_NATURALES,PUNT_SOCIALES_CIUDADANAS,PUNT_INGLES,PUNT_GLOBAL
0,1,1,1,48,51,26,14,67,194
1,1,2,2,77,32,64,48,48,199
2,2,3,3,65,43,63,84,48,268
3,3,4,4,61,45,52,62,49,365
4,3,2,4,68,87,52,16,57,355


## ⏸ Checkpoint del profesor (10 min)

Detengan aquí. El profesor revisa la Tarea 3 con toda la sala: cómo evitar perder filas al hacer joins, qué hacer si aparecen nulos en las FKs.

## Tarea 4 (30 min) — Modelo dimensional del proyecto del grupo

Con tu grupo, diseñen el modelo dimensional para su dataset del proyecto:

1. Identifiquen el **hecho** principal y su granularidad.
2. Identifiquen 3-4 **dimensiones** y justifiquen brevemente cada una.
3. Dibujen el modelo (celda Markdown con diagrama tipo Mermaid, ASCII, o adjunten un PNG de draw.io).

Al final, guarden el diagrama en el repo del grupo bajo `entrega/fase_a_diseño_borrador.pdf`.

### Modelo dimensional — Grupo 7

**Dataset:** estadísticas delictivas de Colombia 2025 (Policía Nacional – DIJIN),
480.091 casos consolidados en `datos/delitos_2025_consolidado.csv`.

#### 1. El hecho y su granularidad

**`hecho_delito` — una fila = un caso delictivo registrado por la Policía
Nacional en 2025.**

Es el grano más fino que entrega la fuente y por eso el que elegimos: el origen
no agrega, entrega un registro por víctima/evento. Bajar de ahí es imposible y
subir (por ejemplo, a municipio × mes) nos cerraría análisis que sí queremos
hacer, como el cruce arma × género × día de la semana.

**Medida:** `CANTIDAD`, que vale 1 en todas las filas. Es una medida aditiva
sobre las cuatro dimensiones: `SUM(CANTIDAD)` cuenta casos en cualquier corte.
Es un *fact table* de tipo **transaccional** sin medidas numéricas propias —
lo que Kimball llama un **factless fact table**, donde el valor analítico está
en el conteo de ocurrencias, no en un monto.

> **Advertencia de grano.** La fuente no trae identificador de caso, así que dos
> filas pueden ser idénticas en los 19 atributos y aun así corresponder a
> eventos distintos (dos hurtos a hombres adultos el mismo día en Bogotá sin
> arma). Por eso el hecho lleva la llave surrogate `ID_CASO`: deja explícito que
> **no se debe aplicar `drop_duplicates()`**, que borraría el 63 % de los datos.

#### 2. Las dimensiones

| Dimensión | PK | Atributos | Filas | Justificación |
|---|---|---|---:|---|
| `dim_tiempo` | `tiempo_id` | FECHA_HECHO, ANIO, TRIMESTRE, MES, NOMBRE_MES, DIA, DIA_SEMANA, ES_FIN_SEMANA | 365 | Es la dimensión conforme por excelencia. Con la jerarquía año → trimestre → mes → día respondemos estacionalidad, y `DIA_SEMANA` / `ES_FIN_SEMANA` son claves aquí: los delitos se concentran en fin de semana (sábado 73.109 vs martes 65.383). |
| `dim_geografia` | `geo_id` | CODIGO_DANE, CODIGO_MUNICIPIO, MUNICIPIO, CODIGO_DEPTO, DEPARTAMENTO, ES_CAPITAL | 1.500 | Jerarquía departamento → municipio → centro poblado, alineada con la codificación DIVIPOLA del DANE. Permite *drill-down* desde el mapa nacional hasta el corregimiento y, sobre todo, **unir con datos externos de población** para calcular tasas por 100.000 habitantes, que es la métrica con la que realmente se comparan territorios. |
| `dim_delito` | `delito_id` | DELITO, CATEGORIA_DELITO | 8 | Jerarquía categoría → delito (Patrimonio → Hurto a personas). Es la dimensión que da sentido a la consolidación: los 8 archivos separados del origen se vuelven un solo eje analítico comparable. |
| `dim_circunstancia` | `circ_id` | ARMA_MEDIO, GENERO, GRUPO_EDAD | 90 | **Junk dimension.** Son tres atributos categóricos de baja cardinalidad (22 × 2 × 3) que describen las circunstancias del caso y el perfil de la víctima. Crear tres dimensiones de 22, 2 y 3 filas ensuciaría el esquema; una sola de 90 combinaciones observadas es el patrón estándar de Kimball para este caso. |

**Lo que deliberadamente NO es dimensión:** no existe `dim_victima` con una fila
por persona. La fuente es anónima —solo sabemos género y rango etario— así que
una dimensión de víctima tendría un grano idéntico al del hecho y ningún valor
analítico. Sus dos atributos viven en `dim_circunstancia`.

**Manejo de nulos:** una bodega no deja FKs nulas. Los nulos de `ARMA_MEDIO`
(2,89 %), `GENERO` (0,31 %) y `GRUPO_EDAD` (0,18 %) se materializan como el
miembro explícito `DESCONOCIDO` dentro de `dim_circunstancia`, de modo que esos
casos siguen contándose en los totales en lugar de desaparecer en los JOINs.

#### 3. Diagrama

```mermaid
erDiagram
    dim_tiempo {
        int  tiempo_id      PK
        date FECHA_HECHO
        int  ANIO
        int  TRIMESTRE
        int  MES
        text NOMBRE_MES
        int  DIA
        text DIA_SEMANA
        bool ES_FIN_SEMANA
    }
    dim_geografia {
        int  geo_id             PK
        text CODIGO_DANE
        text CODIGO_MUNICIPIO
        text MUNICIPIO
        text CODIGO_DEPTO
        text DEPARTAMENTO
        bool ES_CAPITAL
    }
    dim_delito {
        int  delito_id        PK
        text DELITO
        text CATEGORIA_DELITO
    }
    dim_circunstancia {
        int  circ_id      PK
        text ARMA_MEDIO
        text GENERO
        text GRUPO_EDAD
    }
    hecho_delito {
        int ID_CASO    PK
        int tiempo_id  FK
        int geo_id     FK
        int delito_id  FK
        int circ_id    FK
        int CANTIDAD
    }

    dim_tiempo        ||--o{ hecho_delito : "cuándo"
    dim_geografia     ||--o{ hecho_delito : "dónde"
    dim_delito        ||--o{ hecho_delito : "qué"
    dim_circunstancia ||--o{ hecho_delito : "cómo / a quién"
```

```
                          dim_tiempo (365)
                       tiempo_id PK, FECHA_HECHO,
                       ANIO, TRIMESTRE, MES, NOMBRE_MES,
                       DIA, DIA_SEMANA, ES_FIN_SEMANA
                                  ▲
                                  │ tiempo_id
                                  │
  dim_geografia (1.500) ────► hecho_delito (480.091) ◄──── dim_delito (8)
  geo_id PK,                 ID_CASO   PK                  delito_id PK,
  CODIGO_DANE,               tiempo_id FK → dim_tiempo     DELITO,
  CODIGO_MUNICIPIO,          geo_id    FK → dim_geografia  CATEGORIA_DELITO
  MUNICIPIO,                 delito_id FK → dim_delito
  CODIGO_DEPTO,              circ_id   FK → dim_circunst.
  DEPARTAMENTO,              CANTIDAD  (medida, =1)
  ES_CAPITAL                        │
                                    │ circ_id
                                    ▼
                          dim_circunstancia (90)
                       circ_id PK, ARMA_MEDIO,
                       GENERO, GRUPO_EDAD
```

#### 4. Estado de implementación

El modelo **ya está construido y cargado**, no solo diseñado. El script
`scripts/etl_bodega_delitos.py` del repositorio del grupo lo genera y lo carga a
SQLite en `datos/delitos_2025.db`, con las mismas validaciones del laboratorio:

```
dim_tiempo               365 filas
dim_geografia          1.500 filas
dim_delito                 8 filas
dim_circunstancia         90 filas
hecho_delito         480.091 filas  (480.091 esperadas) — 4 FKs sin nulos ✓
```

El diagrama se entrega también en `entrega/fase_a_diseño_borrador.pdf`.